# 00 - Simple Agentic RAG

This notebook is the shortest package-first demo of the project. It uses the reusable code in `src/agentic_rag` instead of redefining router, retriever, graph, or pipeline functions inside the notebook.


## Setup

Dependencies are managed by the project environment. From the project root, run `uv sync --extra dev` once and use that environment as the notebook kernel.


In [ ]:
import os
import sys
from pathlib import Path

import chromadb
import pandas as pd
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from agentic_rag.constants import SourceType  # noqa: E402
from agentic_rag.ingestion import ensure_chroma_collections  # noqa: E402
from agentic_rag.llm import OpenAITextGenerator, StaticTextGenerator  # noqa: E402
from agentic_rag.pipeline import AgenticRAG  # noqa: E402
from agentic_rag.retrievers import ChromaRetriever  # noqa: E402
from agentic_rag.router import QueryRouter  # noqa: E402
from agentic_rag.settings import Settings  # noqa: E402
from agentic_rag.web_search import TavilyWebSearcher  # noqa: E402

pd.set_option("display.max_colwidth", 160)
PROJECT_ROOT


In [ ]:
load_dotenv(PROJECT_ROOT / ".env")

RUN_EXTERNAL_APIS = True
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5-nano")
OPENAI_TIMEOUT = float(os.getenv("OPENAI_TIMEOUT", "60"))
openai_api_key = os.getenv("OPENAI_API_KEY")
tavily_api_key = os.getenv("TAVILY_API_KEY")

{
    "run_external_apis": RUN_EXTERNAL_APIS,
    "has_openai_key": bool(openai_api_key),
    "has_tavily_key": bool(tavily_api_key),
}


## Load Data And Chroma

The notebook reads the project datasets and uses the shared ingestion helper to create or reuse the Chroma collections.


In [ ]:
qna_df = pd.read_csv(PROJECT_ROOT / "datasets/medical_qna_dataset.csv")
device_df = pd.read_csv(PROJECT_ROOT / "datasets/medical_device_manuals_dataset.csv")

pd.DataFrame(
    [
        {"dataset": "medical_qna_dataset", "rows": len(qna_df), "columns": len(qna_df.columns)},
        {"dataset": "medical_device_manuals_dataset", "rows": len(device_df), "columns": len(device_df.columns)},
    ]
)


In [ ]:
settings = Settings(_env_file=None, chroma_path=PROJECT_ROOT / "chroma_db")
client = chromadb.PersistentClient(path=str(settings.chroma_path))

collection_summary = ensure_chroma_collections(client, qna_df, device_df)
collection_summary


## Retrieve Directly

`ChromaRetriever` is the package retrieval facade used by the full pipeline.


In [ ]:
retriever = ChromaRetriever(chroma_path=str(settings.chroma_path), top_k=3)
query = "What are the contraindications for a dialysis machine?"

device_docs = await retriever.retrieve(SourceType.RETRIEVE_DEVICE, query)
pd.DataFrame(
    [
        {
            "doc_id": doc.doc_id,
            "preview": doc.text[:240],
        }
        for doc in device_docs
    ]
)


## Build The Agentic RAG Pipeline

The pipeline composes the package router, retriever, generator, and optional web searcher.


In [ ]:
router = QueryRouter(mode="heuristic")

if RUN_EXTERNAL_APIS and openai_api_key:
    generator = OpenAITextGenerator(api_key=openai_api_key, model=OPENAI_MODEL, timeout=OPENAI_TIMEOUT)
else:
    generator = StaticTextGenerator(
        "Offline demo answer. Set RUN_EXTERNAL_APIS=True and OPENAI_API_KEY to generate a real answer."
    )

web_searcher = TavilyWebSearcher(max_results=1) if RUN_EXTERNAL_APIS and tavily_api_key else None

pipeline = AgenticRAG(
    router=router,
    retriever=retriever,
    generator=generator,
    web_searcher=web_searcher,
)

pipeline


In [ ]:
pipeline_graph = """
graph TD
    Query[User Query] --> Router[QueryRouter]
    Router -->|Retrieve_QnA| QnA[ChromaRetriever: medical_qna]
    Router -->|Retrieve_Device| Device[ChromaRetriever: medical_device_manual]
    Router -->|Web_Search| Web[TavilyWebSearcher optional]
    QnA --> Context[Retrieved Context]
    Device --> Context
    Web --> Context
    Context --> Prompt[AgenticRAG._build_prompt]
    Prompt --> Generator[Text Generator]
    Generator --> Response[RAGResponse]
"""

print(pipeline_graph)


In [ ]:
qna_response = await pipeline.answer("What are the treatments for Kawasaki disease?")

{
    "source": qna_response.source.value,
    "answer": qna_response.answer,
    "context_doc_ids": [doc.doc_id for doc in qna_response.context],
    "top_context_preview": qna_response.context[0].text[:240] if qna_response.context else None,
}


In [ ]:
device_response = await pipeline.answer("What are contraindications for a dialysis machine?")

{
    "source": device_response.source.value,
    "answer": device_response.answer,
    "context_doc_ids": [doc.doc_id for doc in device_response.context],
    "top_context_preview": device_response.context[0].text[:240] if device_response.context else None,
}
